# 04 - Agent v2 (LangGraph) on the golden set

Same questions, same models and same answer prompt as notebook 02, but retrieval is driven by an
agent: the LLM first plans one search per company and fiscal year (it can recognise a company that
the question only describes), the searches run with filter + reranker, a check step may ask for one
more search when a needed fact is missing, and then the model answers from the collected passages.

Run notebook 02 first if you want the comparison table at the end (it needs `results/rag_*.jsonl`
in this session; otherwise the comparison is skipped).

Before running: Runtime > T4 GPU, Colab Secret `HF_TOKEN` (notebook access on). One model by default.

v2 changes: descriptive search queries, an extra search with the original question, and the final
passages are chosen by reranking everything collected against the original question (one passage per
filing is always kept, so a two-year comparison sees both years).


In [ ]:
import os
import subprocess
import sys

REPO = "agentic-rag-eval"
BRANCH = "p1-agentic-rag"
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "-b", BRANCH, f"https://github.com/ilaria-cesa/{REPO}.git"], check=True
    )
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
os.chdir(REPO)
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements.txt

In [ ]:
import gc
import json
import random
import time

import numpy as np
import pandas as pd
import torch
from google.colab import userdata
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer

from src.agent.graph import AgentConfig, build_agent, regex_fallback, run_agent
from src.agent.llm import generate, load_llm
from src.eval.golden import load_golden, validate
from src.eval.metrics import retrieval_metrics, score_row, summarise
from src.retrieval.embed import embed_texts, load_chunks
from src.retrieval.pipeline import Corpus, scoped_search
from src.retrieval.rerank import Reranker
from src.retrieval.scope import filings_of

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
HF_REPO = "IlariaCesa/sec-10k-chunks"
chunks = load_chunks(hf_hub_download(HF_REPO, "chunks_1600.jsonl", repo_type="dataset"))
embeddings = np.load(
    hf_hub_download(HF_REPO, "embeddings_bge-m3_1600.npy", repo_type="dataset")
).astype(np.float32)
assert len(chunks) == len(embeddings)
questions = load_golden("golden/golden_set.jsonl")
errors, _ = validate(questions, chunks)
assert not errors, errors

corpus = Corpus(chunks, embeddings)
filings = filings_of(chunks)
names = {c["ticker"]: c["company"] for c in chunks}
embedder = SentenceTransformer("BAAI/bge-m3", device="cuda", model_kwargs={"dtype": torch.float16})
embedder.max_seq_length = 640
reranker = Reranker()


def search(query, ticker, year, k):
    vector = embed_texts(embedder, [query], progress=False)[0]
    return scoped_search(corpus, vector, query, ticker, year, k, reranker)


def score_fn(question, passages):
    return reranker.score(question, [c["embed_text"] for c in passages])


print(len(chunks), "chunks,", len(questions), "questions")

In [ ]:
def make_llm(tokenizer, model):
    def llm(messages, max_new_tokens):
        return generate(tokenizer, model, messages, max_new_tokens)["text"]

    return llm


MODELS = ["Qwen/Qwen3.5-9B"]  # add "Qwen/Qwen3-8B" to compare; each model adds ~30 minutes
TAG = "agent_v2"
all_rows = {}
for name in MODELS:
    print("=" * 80, "\nLoading", name)
    gc.collect()
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f"free GPU memory: {free_gb:.1f} GB")
    assert free_gb > 9, (
        "GPU memory is taken by an earlier run: Runtime > Restart session, then Run all"
    )
    torch.cuda.reset_peak_memory_stats()
    tokenizer, model, loader = load_llm(name)

    llm = make_llm(tokenizer, model)

    agent = build_agent(
        llm,
        search,
        filings,
        names,
        AgentConfig(),
        default_scope=regex_fallback(filings),
        score_fn=score_fn,
    )
    records = []
    for q in questions:
        start = time.time()
        out = run_agent(agent, q["question"])
        seconds = round(time.time() - start, 1)
        row = score_row(q, out, out["evidence_ids"])
        if q["expected"] == "answer":
            row |= retrieval_metrics(q["evidence_chunk_ids"], out["evidence_ids"])
        row |= {
            "model": name,
            "question": q["question"],
            "answer": out["answer"],
            "cited_ids": out["cited_ids"],
            "had_thinking": out["had_thinking"],
            "evidence_ids": out["evidence_ids"],
            "rounds": out["rounds"],
            "trace": out["trace"],
            "seconds": seconds,
            "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 1),
        }
        records.append(row)
        print(
            q["id"],
            q["type"],
            "refused" if row["refused"] else "answered",
            "ok" if row["behaviour_ok"] else "WRONG",
            f"rounds={out['rounds']} {seconds}s",
        )
    all_rows[name] = records
    short = name.split("/")[-1].lower()
    os.makedirs("results", exist_ok=True)
    with open(f"results/{TAG}_{short}.jsonl", "w") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in records)
    del model, tokenizer, agent, llm
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
pd.set_option("display.width", 200)
columns = [
    "n",
    "behaviour_ok",
    "refused",
    "invalid_citation",
    "cites_evidence",
    "numbers_match",
    "hit@5",
    "all@5",
    "rounds",
    "seconds",
    "peak_vram_gb",
]
overall = {}
for name, records in all_rows.items():
    summary = summarise(records)
    print("\n" + name)
    print(pd.DataFrame(summary).T.reindex(columns=columns).to_string())
    overall[name] = summary
with open(f"results/{TAG}_summary.json", "w") as f:
    json.dump(overall, f, indent=1)

In [ ]:
import glob

for path in sorted(glob.glob("results/rag_filter_rerank_*.jsonl")):
    with open(path) as fh:
        rows = [json.loads(line) for line in fh]
    cols = ["n", "behaviour_ok", "refused", "cites_evidence", "numbers_match", "hit@5", "all@5"]
    print("\nnotebook 02 (single search):", rows[0]["model"])
    print(pd.DataFrame(summarise(rows)).T.reindex(columns=cols).to_string())

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("agent_v2_results", "zip", "results")
files.download("agent_v2_results.zip")

Copy back to the chat: the summary table of the agent, the "loaded with" line, and the notebook 02
table if it printed. The zip has every answer and the agent's step log (`trace`) per question.